# 2. Modelos de Voz (Audio)

TTS: texto → audio.  
STT: audio → texto.  
Traducción: audio → inglés.

## Setup

Instalar SDK.

In [ ]:
%pip install --upgrade openai

## API Key

Usar variable de entorno.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from IPython.display import Audio, display
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

client = OpenAI()
print("Cliente listo")

## 2.1 Text-to-Speech

Parámetros clave:
- `model`: modelo TTS.
- `voice`: voz.
- `input`: texto.
- `response_format`: formato.
- `speed`: velocidad.

In [ ]:
TTS_MODEL = "gpt-4o-mini-tts"
VOICE = "coral"
RESPONSE_FORMAT = "mp3"
SPEED = 1.0

texto = "Hola. Este es un ejemplo de generación de voz usando la API de OpenAI."

output_path = Path("demo_tts.mp3")

request_kwargs = {
    "model": TTS_MODEL,
    "voice": VOICE,
    "input": texto,
    "response_format": RESPONSE_FORMAT,
    "speed": SPEED,
}

# instructions no aplica a tts-1 ni tts-1-hd
if not TTS_MODEL.startswith("tts-1"):
    request_kwargs["instructions"] = "Habla de forma clara, cálida y profesional."

with client.audio.speech.with_streaming_response.create(**request_kwargs) as response:
    response.stream_to_file(output_path)

print(f"Audio guardado en: {output_path}")
display(Audio(str(output_path)))

## Probar voces

Cambiar `voice`.

In [ ]:
voces = ["alloy", "ash", "ballad", "coral", "echo", "fable", "onyx", "nova", "sage", "shimmer", "verse"]

voice_demo = "nova"

voice_path = Path(f"demo_voice_{voice_demo}.mp3")

with client.audio.speech.with_streaming_response.create(
    model=TTS_MODEL,
    voice=voice_demo,
    input="Esta es una prueba breve de voz.",
    response_format="mp3",
    speed=1.0,
) as response:
    response.stream_to_file(voice_path)

display(Audio(str(voice_path)))

## Buenas prácticas TTS

- Texto corto y claro.
- Separar párrafos.
- Elegir voz por caso de uso.
- Usar `speed` con moderación.
- Guardar archivos con nombres únicos.

## 2.2 Speech-to-Text

Transcribir audio.

In [ ]:
AUDIO_PATH = Path("demo_tts.mp3")

with AUDIO_PATH.open("rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="text",
        language="es",
    )

print(transcription)

## 2.2.1 Transcripción con Whisper

Usar `verbose_json` para timestamps.

In [ ]:
with AUDIO_PATH.open("rb") as audio_file:
    transcription_verbose = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="verbose_json",
        timestamp_granularities=["segment"],
        language="es",
    )

print("Texto:")
print(transcription_verbose.text)

print("\nSegmentos:")
for segment in transcription_verbose.segments:
    print(segment.start, segment.end, segment.text)

## Modelo STT nuevo

Útil para baja latencia.

In [ ]:
with AUDIO_PATH.open("rb") as audio_file:
    transcription_new = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio_file,
        response_format="json",
        language="es",
    )

print(transcription_new.text)

## 2.2.2 Traducción de Audio

Audio en otro idioma → texto en inglés.

In [ ]:
with AUDIO_PATH.open("rb") as audio_file:
    translation = client.audio.translations.create(
        model="whisper-1",
        file=audio_file,
        response_format="json",
    )

print(translation.text)

## Buenas prácticas STT

- Audio limpio.
- Evitar música de fondo.
- Indicar `language` si se conoce.
- Usar `prompt` para nombres raros.
- Dividir audios largos.

## Ejercicio

1. Grabar un audio propio.  
2. Guardarlo como `mi_audio.mp3`.  
3. Transcribirlo.  
4. Traducirlo al inglés.

In [ ]:
mi_audio = Path("mi_audio.mp3")

if mi_audio.exists():
    with mi_audio.open("rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="text",
            language="es",
        )
    print(result)
else:
    print("Coloca un archivo mi_audio.mp3 junto al notebook.")